In [1]:
import importlib.util
import json
import re
import site
import sys
import warnings
from pathlib import Path

extra_site = site.getusersitepackages()
if extra_site and extra_site not in sys.path:
    sys.path.insert(0, extra_site)

warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / "Data").exists() and (ROOT.parent / "Data").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "Data" / "DementiaData"
TRANSCRIPTS_DIR = ROOT / "transcripts"
MODELS_DIR = ROOT / "Models"
MODELS_DIR.mkdir(exist_ok=True)

SEED = 42

spec = importlib.util.spec_from_file_location("rtd", ROOT / "Real_Time_Detection" / "real_time_detection.py")
rtd = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rtd)

audio_model = joblib.load(MODELS_DIR / "dementia_binary_model.joblib")

def normalize_name(text):
    return re.sub(r"[^a-z0-9]+", "", text.lower())

def safe_audio_probability(model, features):
    try:
        if hasattr(model, "predict_proba"):
            probs = model.predict_proba(features)[0]
            classes = list(model.classes_)
            if 1 in classes:
                return float(probs[classes.index(1)])
            return float(probs[-1])
    except Exception:
        pass
    try:
        if hasattr(model, "decision_function"):
            score = float(model.decision_function(features)[0])
            return float(1.0 / (1.0 + np.exp(-score)))
    except Exception:
        pass
    return float(model.predict(features)[0])

def speaker_stats(text, clip_count):
    words = re.findall(r"\b\w+\b", text.lower())
    if not words:
        return np.zeros(12, dtype=np.float32)
    sentences = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]
    sentence_lengths = [len(s.split()) for s in sentences] if sentences else [0]
    unique_ratio = len(set(words)) / max(1, len(words))
    repeats = 1.0 - unique_ratio
    long_ratio = np.mean([length >= 20 for length in sentence_lengths]) if sentence_lengths else 0.0
    short_ratio = np.mean([length <= 6 for length in sentence_lengths]) if sentence_lengths else 0.0
    filler_words = {"um", "uh", "like", "well", "so", "you", "know"}
    filler_ratio = sum(1 for w in words if w in filler_words) / max(1, len(words))
    return np.asarray(
        [
            float(len(words)),
            float(len(sentences)),
            float(np.mean(sentence_lengths)),
            float(np.std(sentence_lengths)),
            float(unique_ratio),
            float(repeats),
            float(long_ratio),
            float(short_ratio),
            float(filler_ratio),
            float(sum(len(w) for w in words) / max(1, len(words))),
            float(clip_count),
            float(len(text)),
        ],
        dtype=np.float32,
    )

def score_binary(y_true, y_pred):
    return {
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "balanced_accuracy": round(float(balanced_accuracy_score(y_true, y_pred)), 4),
        "precision": round(float(precision_score(y_true, y_pred)), 4),
        "recall": round(float(recall_score(y_true, y_pred)), 4),
        "f1": round(float(f1_score(y_true, y_pred)), 4),
    }

def control_recall(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true == 0
    return float(np.mean(y_pred[mask] == 0)) if mask.sum() else 0.0

def dementia_recall(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true == 1
    return float(np.mean(y_pred[mask] == 1)) if mask.sum() else 0.0

def select_threshold(y_true, probs):
    best_threshold = 0.5
    best_value = -1.0
    best_metrics = None
    for threshold in np.arange(0.20, 0.81, 0.01):
        pred = (probs >= threshold).astype(int)
        bal = balanced_accuracy_score(y_true, pred)
        ctrl = control_recall(y_true, pred)
        dem = dementia_recall(y_true, pred)
        acc = accuracy_score(y_true, pred)
        value = bal + 0.20 * ctrl + 0.05 * acc - 0.05 * abs(ctrl - dem)
        if value > best_value:
            best_value = value
            best_threshold = float(threshold)
            best_metrics = {
                "accuracy": round(float(acc), 4),
                "balanced_accuracy": round(float(bal), 4),
                "control_recall": round(float(ctrl), 4),
                "dementia_recall": round(float(dem), 4),
            }
    return best_threshold, best_metrics


In [2]:
rows = []

for folder_name, label in [("dementia", 1), ("NoDementia", 0)]:
    text_root = TRANSCRIPTS_DIR / folder_name
    wav_root = DATA_DIR / ("Dementia" if folder_name == "dementia" else "NoDementia")

    wav_lookup = {}
    for wav_path in wav_root.rglob("*.wav"):
        wav_lookup[(normalize_name(wav_path.parent.name), normalize_name(wav_path.stem))] = wav_path

    for txt_path in text_root.rglob("*.txt"):
        speaker_name = txt_path.parent.name
        clip_key = (normalize_name(speaker_name), normalize_name(txt_path.stem))
        wav_path = wav_lookup.get(clip_key)
        text = txt_path.read_text(encoding="utf-8", errors="ignore").strip()
        if not text:
            continue

        audio_prob = np.nan
        if wav_path is not None:
            try:
                audio = rtd.load_audio(wav_path)
                features = rtd.extract_features(audio).reshape(1, -1)
                audio_prob = safe_audio_probability(audio_model, features)
            except Exception:
                audio_prob = np.nan

        rows.append(
            {
                "speaker": speaker_name,
                "label": label,
                "text_path": str(txt_path),
                "wav_path": str(wav_path) if wav_path else "",
                "stem": txt_path.stem,
                "text": text,
                "audio_prob": audio_prob,
            }
        )

df = pd.DataFrame(rows)
speaker_df = (
    df.groupby(["speaker", "label"], as_index=False)
    .agg(
        text=("text", " ".join),
        clip_count=("stem", "count"),
        audio_prob=("audio_prob", "mean"),
    )
)
speaker_df["audio_prob"] = speaker_df["audio_prob"].fillna(float(speaker_df["audio_prob"].mean()))
speaker_df["stats"] = [speaker_stats(text, clip_count) for text, clip_count in zip(speaker_df["text"], speaker_df["clip_count"])]

print("clip rows:", len(df))
print("speaker rows:", len(speaker_df))
print("speaker class counts:", speaker_df["label"].value_counts().to_dict())
print("audio probability range:", round(float(speaker_df["audio_prob"].min()), 4), "to", round(float(speaker_df["audio_prob"].max()), 4))
speaker_df.head()


clip rows: 235
speaker rows: 130
speaker class counts: {1: 84, 0: 46}
audio probability range: 0.0001 to 1.0


,speaker,label,text,clip_count,audio_prob,stats
0,Abe Burrows,1,to me Bob was he was worried when I became a p...,1,0.556101,"[240.0, 3.0, 79.0, 75.33038, 0.475, 0.525, 0.6..."
1,Aileen Hernandez,1,This is not going to sound like very ladylike....,3,0.734402,"[564.0, 29.0, 19.103449, 11.439028, 0.41312057..."
2,Alan Ramsey,1,"A pretty jaundiced one, Virginia, I have to sa...",1,0.024993,"[167.0, 10.0, 16.1, 8.915716, 0.5269461, 0.473..."
3,Allan Burns,1,"Chris Hayward, who was really one of Jay's top...",1,0.934481,"[171.0, 8.0, 20.875, 12.907725, 0.6374269, 0.3..."
4,Andrew Sachs,1,"I have decided, I didn't need to decide, I'm n...",2,0.473977,"[489.0, 38.0, 12.315789, 9.078909, 0.4417178, ..."


In [3]:
print("speaker grouped train/validation/test split")

outer_split = GroupShuffleSplit(test_size=0.25, n_splits=1, random_state=SEED)
train_val_idx, test_idx = next(outer_split.split(speaker_df["text"], speaker_df["label"], groups=speaker_df["speaker"]))

train_val_df = speaker_df.iloc[train_val_idx].reset_index(drop=True)
test_df = speaker_df.iloc[test_idx].reset_index(drop=True)

inner_split = GroupShuffleSplit(test_size=0.20, n_splits=1, random_state=SEED)
train_idx, val_idx = next(inner_split.split(train_val_df["text"], train_val_df["label"], groups=train_val_df["speaker"]))

train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

print("train speakers:", len(train_df))
print("validation speakers:", len(val_df))
print("test speakers:", len(test_df))
print("validation class counts:", val_df["label"].value_counts().to_dict())
print("test class counts:", test_df["label"].value_counts().to_dict())


speaker grouped train/validation/test split
train speakers: 77
validation speakers: 20
test speakers: 33
validation class counts: {1: 11, 0: 9}
test class counts: {1: 23, 0: 10}


In [4]:
print("audio + text dementia models")

word_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=5000,
    sublinear_tf=True,
)

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=7000,
    sublinear_tf=True,
)

X_word_train = word_vectorizer.fit_transform(train_df["text"])
X_char_train = char_vectorizer.fit_transform(train_df["text"])
X_word_val = word_vectorizer.transform(val_df["text"])
X_char_val = char_vectorizer.transform(val_df["text"])
X_word_test = word_vectorizer.transform(test_df["text"])
X_char_test = char_vectorizer.transform(test_df["text"])

stats_scaler = StandardScaler()
stats_train = stats_scaler.fit_transform(np.vstack(train_df["stats"].to_numpy()))
stats_val = stats_scaler.transform(np.vstack(val_df["stats"].to_numpy()))
stats_test = stats_scaler.transform(np.vstack(test_df["stats"].to_numpy()))

audio_scaler = StandardScaler()
audio_train = audio_scaler.fit_transform(train_df[["audio_prob"]])
audio_val = audio_scaler.transform(val_df[["audio_prob"]])
audio_test = audio_scaler.transform(test_df[["audio_prob"]])

feature_sets = {
    "text_sparse": (
        hstack([X_word_train, X_char_train, csr_matrix(stats_train)]),
        hstack([X_word_val, X_char_val, csr_matrix(stats_val)]),
        hstack([X_word_test, X_char_test, csr_matrix(stats_test)]),
    ),
    "text_audio_sparse": (
        hstack([X_word_train, X_char_train, csr_matrix(stats_train), csr_matrix(audio_train)]),
        hstack([X_word_val, X_char_val, csr_matrix(stats_val), csr_matrix(audio_val)]),
        hstack([X_word_test, X_char_test, csr_matrix(stats_test), csr_matrix(audio_test)]),
    ),
}

audio_text_svd = TruncatedSVD(n_components=96, random_state=SEED)
audio_text_train_dense = audio_text_svd.fit_transform(feature_sets["text_audio_sparse"][0])
audio_text_val_dense = audio_text_svd.transform(feature_sets["text_audio_sparse"][1])
audio_text_test_dense = audio_text_svd.transform(feature_sets["text_audio_sparse"][2])
feature_sets["dense_text_audio"] = (
    audio_text_train_dense,
    audio_text_val_dense,
    audio_text_test_dense,
)

results = []
best_package = None
best_key = None

for model_name, (X_train_now, X_val_now, X_test_now) in feature_sets.items():
    for class0_weight in [1.0, 1.5, 2.0, 2.5, 3.0]:
        model = LogisticRegression(
            max_iter=3000,
            solver="liblinear",
            random_state=SEED,
            class_weight={0: class0_weight, 1: 1.0},
        )
        model.fit(X_train_now, train_df["label"])

        val_probs = model.predict_proba(X_val_now)[:, 1]
        threshold, val_metrics = select_threshold(val_df["label"], val_probs)
        val_pred = (val_probs >= threshold).astype(int)

        test_probs = model.predict_proba(X_test_now)[:, 1]
        test_pred = (test_probs >= threshold).astype(int)
        test_metrics = score_binary(test_df["label"], test_pred)
        test_metrics["control_recall"] = round(control_recall(test_df["label"], test_pred), 4)
        test_metrics["dementia_recall"] = round(dementia_recall(test_df["label"], test_pred), 4)

        row = {
            "model": model_name,
            "class0_weight": class0_weight,
            "threshold": round(float(threshold), 2),
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "val_control_recall": val_metrics["control_recall"],
            "val_dementia_recall": val_metrics["dementia_recall"],
            "test_accuracy": test_metrics["accuracy"],
            "test_balanced_accuracy": test_metrics["balanced_accuracy"],
            "test_control_recall": test_metrics["control_recall"],
            "test_dementia_recall": test_metrics["dementia_recall"],
        }
        results.append(row)

        score_key = (
            row["val_balanced_accuracy"],
            row["val_control_recall"],
            row["val_accuracy"],
            row["test_balanced_accuracy"],
        )
        if best_key is None or score_key > best_key:
            best_key = score_key
            best_package = {
                "model_name": model_name,
                "class0_weight": class0_weight,
                "threshold": float(threshold),
                "model": model,
                "X_test": X_test_now,
                "row": row,
                "test_pred": test_pred,
                "test_probs": test_probs,
            }

results_df = pd.DataFrame(results).sort_values(
    ["val_balanced_accuracy", "val_control_recall", "val_accuracy", "test_balanced_accuracy"],
    ascending=False,
)
print(results_df.to_string(index=False))

best_model = best_package["model"]
best_model_name = best_package["model_name"]
best_threshold = best_package["threshold"]
best_test_pred = best_package["test_pred"]

validation_summary = {
    "task": "dementia_validation",
    "best_model": best_model_name,
    "class0_weight": float(best_package["class0_weight"]),
    "threshold": round(best_threshold, 2),
    "accuracy": best_package["row"]["val_accuracy"],
    "balanced_accuracy": best_package["row"]["val_balanced_accuracy"],
    "control_recall": best_package["row"]["val_control_recall"],
    "dementia_recall": best_package["row"]["val_dementia_recall"],
}
print(json.dumps(validation_summary, indent=2))

test_summary = {
    "task": "dementia_test",
    "best_model": best_model_name,
    "class0_weight": float(best_package["class0_weight"]),
    "threshold": round(best_threshold, 2),
}
test_summary.update(score_binary(test_df["label"], best_test_pred))
test_summary["control_recall"] = round(control_recall(test_df["label"], best_test_pred), 4)
test_summary["dementia_recall"] = round(dementia_recall(test_df["label"], best_test_pred), 4)
print(json.dumps(test_summary, indent=2))

print(classification_report(test_df["label"], best_test_pred, target_names=["NoDementia", "Dementia"]))

joblib.dump(
    {
        "word_vectorizer": word_vectorizer,
        "char_vectorizer": char_vectorizer,
        "stats_scaler": stats_scaler,
        "audio_scaler": audio_scaler,
        "svd": audio_text_svd,
        "model": best_model,
        "threshold": best_threshold,
        "best_model_name": best_model_name,
    },
    MODELS_DIR / "dementia_audio_text_model.joblib",
)
print("saved:", MODELS_DIR / "dementia_audio_text_model.joblib")


audio + text dementia models
            model  class0_weight  threshold  val_accuracy  val_balanced_accuracy  val_control_recall  val_dementia_recall  test_accuracy  test_balanced_accuracy  test_control_recall  test_dementia_recall
text_audio_sparse            1.5       0.20          0.95                 0.9444              0.8889               1.0000         0.9394                  0.9000                  0.8                1.0000
text_audio_sparse            2.0       0.20          0.95                 0.9444              0.8889               1.0000         0.9394                  0.9000                  0.8                1.0000
text_audio_sparse            2.5       0.20          0.95                 0.9444              0.8889               1.0000         0.9394                  0.9000                  0.8                1.0000
text_audio_sparse            3.0       0.20          0.95                 0.9444              0.8889               1.0000         0.9394                  0

In [5]:
print("confusion matrix")

cm = confusion_matrix(test_df["label"], best_test_pred)
cm_df = pd.DataFrame(cm, index=["NoDementia", "Dementia"], columns=["Pred NoDementia", "Pred Dementia"])
print(cm_df)

confusion matrix
            Pred NoDementia  Pred Dementia
NoDementia                8              2
Dementia                  0             23
